In [ ]:
from ultralytics import YOLO
from IPython.display import display, Image

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("apikeys.env")                 # reads ./.env by default
ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY")
os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY
print(ROBOFLOW_API_KEY)

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

In [ ]:
!pwd

In [ ]:
project = rf.workspace("large-benchmark-datasets").project("logistics-sz9jr")
dataset = project.version(2).download("yolov8")

In [ ]:
model = YOLO("best.pt")

In [ ]:

# Validate the model
metrics = model.val(data = "/teamspace/studios/this_studio/Workshop2/Logistics-2/data.yaml")
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps  # a list contains map50-95 of each category

In [ ]:
img_path = "/teamspace/studios/this_studio/Workshop2/Logistics-2/test/images/-_10_jpg.rf.7216b8fd23d002354ad6cdd0d7691cd5.jpg"
results = model.predict(source=img_path, conf=0.25, save=True)

In [ ]:
results[0].boxes.xyxy

In [ ]:
results

In [ ]:
Image(filename='runs/detect/predict/-_10_jpg.rf.7216b8fd23d002354ad6cdd0d7691cd5.jpg', height=600)

In [ ]:
!pwd

In [ ]:
metrics = model.val(
    data="/teamspace/studios/this_studio/Workshop2/Logistics-2/data.yaml",  # path to data.yaml
    split="test",   # run evaluation on test split instead of val
    imgsz=640,
    conf=0.25
)

print(metrics)  # will include mAP50, mAP50-95, precision, recall, etc.

In [ ]:
def encode_response(results):
    """
    Convert Ultralytics YOLO results (list of Results) into JSON-friendly dict.
    Uses class names from the results array itself.
    """
    detections = []
    if not results:
        return {"detections": detections}

    r = results[0]   # one Results object
    boxes = r.boxes

    if boxes is None or len(boxes) == 0:
        return {"detections": detections}

    cls = boxes.cls.cpu().numpy().astype(int)
    conf = boxes.conf.cpu().numpy()
    xyxy = boxes.xyxy.cpu().numpy()

    # names mapping is in the result object
    names = r.names  

    for class_id, confidence, bbox in zip(cls, conf, xyxy):
        detections.append({
            "class_id": int(class_id),
            "class_name": names[class_id],
            "confidence": float(confidence),
            "bbox": bbox.tolist()  # [x1, y1, x2, y2]
        })

    return {"detections": detections}

In [ ]:
encode_response(results)

In [ ]:
results

In [ ]:
!python client.py --image /teamspace/studios/this_studio/Workshop2/Logistics-2/test/images/-_10_jpg.rf.7216b8fd23d002354ad6cdd0d7691cd5.jpg